### Exercise 1 — Tokenization & Vocabulary
- Create a simple tokenizer and vocabulary.  #
- Convert a sentence into integer token IDs.


In [1]:
# Fill in the blanks

PAD, UNK, CLS = "<pad>", "<unk>", "<cls>"
sentence = "I love artificial intelligence"

# Step 1: Split and lowercase
tokens = [CLS] + sentence.lower().split()

# Step 2: Build a small vocabulary
vocab = sorted(set(tokens + [PAD, UNK]))
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}

# Step 3: Encode the sentence
ids = [stoi.get(tok, stoi[UNK]) for tok in tokens]

print("Tokens:", tokens)
print("Token IDs:", ids)


Tokens: ['<cls>', 'i', 'love', 'artificial', 'intelligence']
Token IDs: [0, 4, 6, 3, 5]


 **Reflect:**  
- Why do we add a `<cls>` token?  
- How would you handle unknown words in new sentences?



### Exercise 2: 
Create sinusoidal positional encodings to inject order information into token embeddings.


In [2]:
import math, torch

# Fill in the blanks

def positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(0, seq_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

pe = positional_encoding(10, 8)
print(pe[:2])


tensor([[0.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00,
         0.0000e+00, 1.0000e+00],
        [8.4147e-01, 5.4030e-01, 9.9833e-02, 9.9500e-01, 9.9998e-03, 9.9995e-01,
         1.0000e-03, 1.0000e+00]])


 **Reflect:**  
- Why do we alternate sine and cosine values?  
- What benefit do sinusoidal encodings have compared to learned positional embeddings?

---


### Exercise 3
- Implement the attention mechanism manually:  
- Softmax(QKᵀ / √dₖ) × V


In [3]:
import torch, math
torch.manual_seed(0)

# Fill in the blanks

Q = torch.randn(1, 3, 4)   # [batch, tokens, d_model]
K = torch.randn(1, 3, 4)
V = torch.randn(1, 3, 4)

scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(4)
attn = torch.softmax(scores, dim=-1)
out = torch.matmul(attn, V)

print("Attention weights:\n", attn)
print("Output:\n", out)


Attention weights:
 tensor([[[0.4881, 0.3514, 0.1605],
         [0.3947, 0.2935, 0.3119],
         [0.4543, 0.3873, 0.1584]]])
Output:
 tensor([[[ 0.1171,  0.4515, -0.8037, -0.8058],
         [ 0.1295,  0.4572, -1.0491, -0.6166],
         [ 0.1095,  0.3819, -0.7369, -0.8267]]])


 **Reflect:**  
- What would happen if we didn’t scale by √dₖ?  
- Which rows in the attention matrix correspond to which tokens?

---


### Exercise 4:
Simulate 2 attention heads by splitting Q, K, and V manually.


In [4]:
# Fill in the blanks

import torch, math
torch.manual_seed(0)

d_model = 8
num_heads = 2
d_head = d_model // num_heads

x = torch.randn(1, 3, d_model)

# Create projection matrices
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)

Q, K, V = x @ W_q, x @ W_k, x @ W_v

# Split into heads
Q = Q.view(1, 3, num_heads, d_head).transpose(1, 2)
K = K.view(1, 3, num_heads, d_head).transpose(1, 2)
V = V.view(1, 3, num_heads, d_head).transpose(1, 2)

# Compute attention per head
attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_head)
attn = torch.softmax(attn_scores, dim=-1)
out = torch.matmul(attn, V)

print(out.shape)


torch.Size([1, 2, 3, 4])


 **Reflect:**  
- Why do we use multiple heads instead of one?  
- What different relationships could each head learn?

---


### Exercise 5:
Implement a minimal Transformer encoder block using:  
1️. Multi-Head Attention  
2️. Feed-Forward Network  
3️. Residual + Layer Normalization


In [5]:
import torch
import torch.nn as nn

# Fill in the blanks

class MiniEncoder(nn.Module):
    def __init__(self, d_model=8, d_ff=16):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=2, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.mha(x, x, x)
        x = self.ln1(x + attn_out)
        ff_out = self.ff(x)
        return self.ln2(x + ff_out)

# Test block
x = torch.randn(1, 4, 8)
enc = MiniEncoder()
y = enc(x)
print(y.shape)


torch.Size([1, 4, 8])


### 1. Why do we add residual connections?

Residual connections help preserve information from earlier layers and make training deep neural networks easier. They allow gradients to flow more effectively during backpropagation, reducing the vanishing gradient problem. This helps the Transformer learn faster and achieve better performance.

---

### 2. What happens if LayerNorm is removed from the network?

If LayerNorm is removed, training becomes less stable because the activations can grow too large or too small. The model may converge slowly, produce inconsistent results, or fail to learn effectively. Layer Normalization helps maintain a stable distribution of values throughout the network, improving training efficiency and accuracy.
